# RareMind - Agentic Pipeline Demo

End-to-end walkthrough of the agentic rare disease QA pipeline.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from src.utils.config_loader import load_config
config = load_config("../config/config.yaml")
print("Config loaded:", list(config.keys()))

## 2. Generate and Ingest the Pseudo Dataset

In [ ]:
from data.pseudo_dataset.generate_dataset import main as gen_data
gen_data()

In [ ]:
from src.tools.document_processor import DocumentProcessor
from src.tools.vector_store import VectorStoreTool

processor = DocumentProcessor(config=config)
chunks = processor.load_from_json("../data/pseudo_dataset/rare_disease_docs.json", text_key="content")
print(f"Produced {len(chunks)} chunks")

vs = VectorStoreTool(config=config)
vs.build(chunks)
print("Vector store built!")

## 3. Build the Planning Agent

In [ ]:
from src.agents.planning_agent import PlanningAgent
agent = PlanningAgent(config=config)
print("Agent ready!")

## 4. Run Example Queries

In [ ]:
response = agent.run("What is Kaposiform Lymphangiomatosis and how is it treated?")
print(f"Route: {response.route} | Confidence: {response.confidence:.2f}")
print()
print(response.final_answer)

In [ ]:
# Follow-up - should route to history
response2 = agent.run("What side effects should I watch for with that treatment?")
print(f"Route: {response2.route}")
print()
print(response2.final_answer)

## 5. Inspect the Reasoning Trace

In [ ]:
for step in response.trace:
    print(f"Step {step.step} [{step.agent}] {step.action}")
    print(f"  -> {step.result_summary} ({step.duration_ms:.0f}ms)")

## 6. Evaluate the Pipeline

In [ ]:
import json
agent.reset_memory()

with open("../data/pseudo_dataset/eval_questions.json") as f:
    questions = json.load(f)

results = []
for q in questions:
    r = agent.run(q["question"])
    results.append({"question": q["question"], "route": r.route})
    agent.reset_memory()

for r in results:
    print(f"Q: {r['question'][:60]}...")
    print(f"   Route: {r['route']}")
    print()